# COCO Pipeline Test: MaskRCNN + Red Circle + SpLiCE

Testing the full updated pipeline from utils.py on a COCO subset:
1. Load COCO images
2. Run MaskRCNN to detect objects → Bounding Boxes
3. Mark objects: Crop / BBox / Red Circle
4. Generate SpLiCE concept activations
5. Compare results

In [ ]:
import sys
sys.path.insert(0, "/workspaces/Object-Centric-Concept-Bottlenecks")

import torch
import open_clip
import splice
from splice.model import SPLICE
from PIL import Image, ImageDraw
import torchvision.transforms.functional as TF
import torchvision.transforms.v2 as T
from torchvision.models.detection import maskrcnn_resnet50_fpn
import matplotlib.pyplot as plt
import os

from utils import (
    generate_object_propsals,
    gather_object_embeddings,
    draw_box_on_image,
    draw_circle_on_image,
)

In [ ]:
# MaskRCNN
maskrcnn = maskrcnn_resnet50_fpn(pretrained=True)
maskrcnn.eval()
print("MaskRCNN loaded")

# SpLiCE
concepts = torch.load("/root/.cache/splice/embeddings/open_clip:ViT-B-32_laion_10000_embeddings.pt", map_location="cpu")
image_mean = torch.load("/root/.cache/splice/means/open_clip_ViT-B-32_image.pt", map_location="cpu")
clip_backbone, _, splice_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
clip_backbone.eval()
splice_model = SPLICE(image_mean=image_mean, dictionary=concepts, clip=clip_backbone, device="cpu", l1_penalty=0.25, return_weights=True)
splice_model.eval()
vocab = splice.get_vocabulary("laion", vocabulary_size=10000)
print("SpLiCE loaded")

In [ ]:
IMG_DIR = "/workspaces/Object-Centric-Concept-Bottlenecks/datasets/coco/images/val2017"

# Pick 10 images
img_files = sorted(os.listdir(IMG_DIR))[:10]

# Transform for MaskRCNN (needs float tensor, no normalization)
to_tensor = T.Compose([T.Resize((640, 640)), T.ToTensor()])

images = []
image_pils = []
for fname in img_files:
    pil = Image.open(os.path.join(IMG_DIR, fname)).convert("RGB")
    image_pils.append(pil)
    images.append(to_tensor(pil))

print(f"Loaded {len(images)} images")
print(f"Image tensor shape: {images[0].shape}")

# Show first 4 images
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for i in range(4):
    axes[i].imshow(image_pils[i])
    axes[i].set_title(img_files[i][:15])
    axes[i].axis("off")
plt.suptitle("First 4 COCO images", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# MaskRCNN braucht eine Liste von Tensoren
with torch.no_grad():
    boxes_list, scores_list = generate_object_propsals(
        images,
        maskrcnn,
        min_size=0.02,
        max_size=0.85,
        iou_threshold=0.5,
        minimum_score=0.3,
    )

for i, (boxes, scores) in enumerate(zip(boxes_list, scores_list)):
    print(f"Image {i+1} ({img_files[i][:20]}): {len(boxes)} objects detected")


In [ ]:
# Zeige für die ersten 4 Bilder: BBox + Circle nebeneinander
for i in range(4):
    boxes = boxes_list[i]
    
    if len(boxes) == 0:
        print(f"Image {i+1}: no objects detected")
        continue

    fig, axes = plt.subplots(1, 2, figsize=(7, 4))

    # BBox — erstes erkanntes Objekt
    bbox_img = draw_box_on_image(images[i], boxes[0])
    axes[0].imshow(bbox_img)
    axes[0].set_title(f"BBox (object 1)")
    axes[0].axis("off")

    # Circle — erstes erkanntes Objekt
    circle_img = draw_circle_on_image(images[i], boxes[0])
    axes[1].imshow(circle_img)
    axes[1].set_title(f"Red Circle (object 1)")
    axes[1].axis("off")

    plt.suptitle(f"Image {i+1}: {img_files[i]}", fontsize=11)
    plt.tight_layout()
    plt.show()


## Image 3: Full Pipeline Walkthrough

Step 1: Visualize all 5 detected objects with Red Circle

Step 2: SpLiCE concepts for full image + each object separately  

Step 3: Sum aggregation over all objects + full image

In [ ]:
img_idx = 2  # Image 3
boxes  = boxes_list[img_idx]
scores = scores_list[img_idx]
image  = images[img_idx]

num_objs = min(5, len(boxes))

fig, axes = plt.subplots(1, num_objs + 1, figsize=(4 * (num_objs + 1), 4))

# Original
axes[0].imshow(image_pils[img_idx])
axes[0].set_title("Original")
axes[0].axis("off")

# Red Circle für jedes Objekt
for j in range(num_objs):
    circle_img = draw_circle_on_image(image, boxes[j])
    axes[j+1].imshow(circle_img)
    axes[j+1].set_title(f"Object {j+1}\nscore: {scores[j]:.2f}")
    axes[j+1].axis("off")

plt.suptitle(f"Image 3: Red Circle per object", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def get_emb(image_pil):
    img_t = splice_preprocess(image_pil).unsqueeze(0)
    with torch.no_grad():
        emb, _ = splice_model.encode_image(img_t)
    return emb.squeeze(0)

def top_concepts(emb, vocab, top_k=5):
    idx = emb.argsort(descending=True)[:top_k]
    return [(vocab[i], round(emb[i].item(), 3)) for i in idx]

# Full image
full_emb = get_emb(image_pils[img_idx])
full_top = top_concepts(full_emb, vocab, top_k=5)

print(f"{'Full Image':<35}")
print("-" * 35)
for concept, val in full_top:
    print(f"  {concept:<25} {val:.3f}")

# Jedes Objekt einzeln
print()
for j in range(num_objs):
    circle_pil = draw_circle_on_image(image, boxes[j])
    obj_emb = get_emb(circle_pil)
    obj_top = top_concepts(obj_emb, vocab, top_k=5)

    print(f"Object {j+1} (score: {scores[j]:.2f})")
    print("-" * 35)
    for concept, val in obj_top:
        print(f"  {concept:<25} {val:.3f}")
    print()

In [ ]:
# Stack alle Objekt-Embeddings
obj_embs = torch.stack([
    get_emb(draw_circle_on_image(image, boxes[j]))
    for j in range(num_objs)
])

# Sum: full image + alle Objekte
sum_agg = full_emb + obj_embs.sum(dim=0)
sum_top = top_concepts(sum_agg, vocab, top_k=10)

print("Sum Aggregation: Full Image + 4 Objects (Red Circle)")
print("-" * 45)
for concept, val in sum_top:
    print(f"  {concept:<25} {val:.3f}")

# Vergleich: was kommt neu hinzu vs. Full Image alone
full_concepts_set   = {c for c, _ in full_top}
sum_concepts_set    = {c for c, _ in sum_top}
new_concepts        = sum_concepts_set - full_concepts_set

print(f"\nNew concepts after aggregation: {new_concepts}")


## Vergleich Crop / BBox / Circle + Sum für Image 3 - Bedroom

In [ ]:
splice_transform = T.Compose([
    T.Resize(size=224, interpolation=T.InterpolationMode.BICUBIC, antialias=True),
    T.CenterCrop(size=(224, 224)),
    T.Normalize(mean=(0.48145466, 0.4578275, 0.40821073),
                std=(0.26862954, 0.26130258, 0.27577711)),
])

kwargs = dict(
    boxes=boxes_list[img_idx],
    scores=scores_list[img_idx],
    min_score=0.3,
    num_objects=5,
    image=images[img_idx],
    device="cpu",
    splice_model=splice_model,
    embedding_dim=10000,
    transform=splice_transform,
)

crop_embs,   _ = gather_object_embeddings(**kwargs, draw_boxes=False, draw_circles=False)
bbox_embs,   _ = gather_object_embeddings(**kwargs, draw_boxes=True,  draw_circles=False)
circle_embs, _ = gather_object_embeddings(**kwargs, draw_boxes=False, draw_circles=True)

# Sum + full image
crop_sum   = full_emb + crop_embs.sum(dim=0)
bbox_sum   = full_emb + bbox_embs.sum(dim=0)
circle_sum = full_emb + circle_embs.sum(dim=0)

# Top concepts
crop_top   = top_concepts(crop_sum,   vocab, top_k=10)
bbox_top   = top_concepts(bbox_sum,   vocab, top_k=10)
circle_top = top_concepts(circle_sum, vocab, top_k=10)

print(f"{'Crop + Sum':<30} {'BBox + Sum':<30} {'Circle + Sum':<30}")
print("-" * 90)
for i in range(10):
    c = f"{crop_top[i][0]} ({crop_top[i][1]:.3f})"
    b = f"{bbox_top[i][0]} ({bbox_top[i][1]:.3f})"
    r = f"{circle_top[i][0]} ({circle_top[i][1]:.3f})"
    print(f"{c:<30} {b:<30} {r:<30}")


## Vergleich Crop / BBox / Circle + Count für Image 3 - Bedroom

In [ ]:

from utils import obj_enc_aggregation

crop_embs,   _ = gather_object_embeddings(**kwargs, draw_boxes=False, draw_circles=False)
bbox_embs,   _ = gather_object_embeddings(**kwargs, draw_boxes=True,  draw_circles=False)
circle_embs, _ = gather_object_embeddings(**kwargs, draw_boxes=False, draw_circles=True)

# obj_enc_aggregation expects (batch, num_objects, dim) and (batch, dim)
full_emb_batch   = full_emb.unsqueeze(0)        # (1, 10000)
crop_batch       = crop_embs.unsqueeze(0)        # (1, N, 10000)
bbox_batch       = bbox_embs.unsqueeze(0)
circle_batch     = circle_embs.unsqueeze(0)

crop_count   = obj_enc_aggregation("count", crop_batch,   full_emb_batch).squeeze(0)
bbox_count   = obj_enc_aggregation("count", bbox_batch,   full_emb_batch).squeeze(0)
circle_count = obj_enc_aggregation("count", circle_batch, full_emb_batch).squeeze(0)

# Top concepts by activation count
def top_count_concepts(emb, vocab, top_k=10):
    idx = emb.argsort(descending=True)[:top_k]
    return [(vocab[i], int(emb[i].item())) for i in idx]

crop_top   = top_count_concepts(crop_count,   vocab)
bbox_top   = top_count_concepts(bbox_count,   vocab)
circle_top = top_count_concepts(circle_count, vocab)

print(f"{'Crop + Count':<30} {'BBox + Count':<30} {'Circle + Count':<30}")
print("(value = how many sources activated this concept)")
print("-" * 90)
for i in range(10):
    c = f"{crop_top[i][0]} ({crop_top[i][1]})"
    b = f"{bbox_top[i][0]} ({bbox_top[i][1]})"
    r = f"{circle_top[i][0]} ({circle_top[i][1]})"
    print(f"{c:<30} {b:<30} {r:<30}")


## Concept Quality Metric

For each image: does the correct COCO category appear in the Top-10 concepts?
Comparing Crop vs. BBox vs. Circle with Sum aggregation.

In [ ]:
import json

ANN_PATH = "/workspaces/Object-Centric-Concept-Bottlenecks/datasets/coco/annotations/instances_val2017.json"
with open(ANN_PATH) as f:
    coco_data = json.load(f)

# Build lookup: image filename → category names
cat_id_to_name = {cat["id"]: cat["name"] for cat in coco_data["categories"]}
img_id_to_fname = {img["id"]: img["file_name"] for img in coco_data["images"]}
fname_to_categories = {}
for ann in coco_data["annotations"]:
    fname = img_id_to_fname[ann["image_id"]]
    cat = cat_id_to_name[ann["category_id"]]
    fname_to_categories.setdefault(fname, set()).add(cat)

# Show ground truth labels for our 10 images
for fname in img_files:
    cats = fname_to_categories.get(fname, set())
    print(f"{fname}: {cats}")

In [ ]:
def concept_rank(emb, vocab, target_words, top_k=20):
    top_idx = emb.argsort(descending=True)[:top_k]
    top_words = [(vocab[i], emb[i].item()) for i in top_idx]
    
    for rank, (word, score) in enumerate(top_words):
        for target in target_words:
            if target in word or word in target:
                return rank + 1, word, score
    return None, None, None

print(f"{'Image':<12} {'GT Label':<18} {'Crop Rank':<12} {'BBox Rank':<12} {'Circle Rank':<12}")
print("-" * 70)

for i, fname in enumerate(img_files):
    boxes  = boxes_list[i]
    scores = scores_list[i]
    cats   = fname_to_categories.get(fname, set())

    if len(boxes) == 0:
        print(f"{fname[:11]:<12} {'(no objects)':<18}")
        continue

    full_emb_i = get_emb(image_pils[i])

    crop_embs,   _ = gather_object_embeddings(boxes, scores, 0.3, 5, images[i], "cpu", splice_model, 10000, splice_transform, draw_boxes=False, draw_circles=False)
    bbox_embs,   _ = gather_object_embeddings(boxes, scores, 0.3, 5, images[i], "cpu", splice_model, 10000, splice_transform, draw_boxes=True,  draw_circles=False)
    circle_embs, _ = gather_object_embeddings(boxes, scores, 0.3, 5, images[i], "cpu", splice_model, 10000, splice_transform, draw_boxes=False, draw_circles=True)

    crop_sum   = full_emb_i + crop_embs.sum(dim=0)
    bbox_sum   = full_emb_i + bbox_embs.sum(dim=0)
    circle_sum = full_emb_i + circle_embs.sum(dim=0)

    for cat in sorted(cats):
        cr, cw, cs = concept_rank(crop_sum,   vocab, [cat])
        br, bw, bs = concept_rank(bbox_sum,   vocab, [cat])
        rr, rw, rs = concept_rank(circle_sum, vocab, [cat])

        cr_str = f"#{cr} ({cw})"   if cr else "not found"
        br_str = f"#{br} ({bw})"   if br else "not found"
        rr_str = f"#{rr} ({rw})"   if rr else "not found"

        print(f"{fname[:11]:<12} {cat:<18} {cr_str:<22} {br_str:<22} {rr_str:<22}")
    print()
